# 01 — Anomera: Exploratory Data Analysis

**Objective:** understand the raw ULB dataset before any model fitting. Do not use the held-out test split for tuning.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA = Path('../data/raw/creditcard.csv')
assert DATA.exists(), 'Download the dataset first; see README.md'
df = pd.read_csv(DATA)
print('Shape:', df.shape)
display(df.head())


## Schema and data quality
Check feature types, nulls, duplicates and label values. Never silently drop records.


In [ ]:
expected = ['Time'] + [f'V{i}' for i in range(1,29)] + ['Amount','Class']
assert set(df.columns) == set(expected)
assert df['Class'].isin([0,1]).all()
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'unique':df.nunique()}))
print('Exact duplicate rows:',df.duplicated().sum())
print('Time range:',df.Time.min(),df.Time.max())


## Class imbalance
Accuracy alone is misleading when almost all transactions are legitimate.


In [ ]:
counts = df['Class'].value_counts().sort_index()
print(pd.DataFrame({'count':counts,'share':counts/len(df)}))
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(['Legitimate','Fraud'],[counts.get(0,0),counts.get(1,0)])
ax.set_ylabel('Transactions');ax.set_title('Class distribution');plt.show()


## Amount and time exploration
Inspect skew and distributions without treating correlations as causal explanations.


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for cls,label in [(0,'Legitimate'),(1,'Fraud')]:
    ax.hist(np.log1p(df.loc[df.Class==cls,'Amount']),bins=60,alpha=.55,density=True,label=label)
ax.set_xlabel('log(1 + Amount)');ax.set_ylabel('Density');ax.legend();plt.show()
display(df.groupby('Class')[['Amount','Time']].agg(['median','mean','max']))


In [ ]:
hour = np.floor(df['Time']/3600).astype(int)
summary = df.groupby(hour).agg(transactions=('Class','size'),frauds=('Class','sum'))
summary['fraud_rate'] = summary['frauds']/summary['transactions']
display(summary.head(10))
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(summary.index,summary['fraud_rate']);ax.set(xlabel='Elapsed hour (not clock hour)',ylabel='Fraud rate',title='Fraud prevalence over time');plt.show()


## Chronological holdout sanity checks
Generate splits using `python scripts/prepare_data.py`. **Do not train or tune on test data.** Verify class counts and boundary ties before model development.


In [ ]:
import json
manifest_path = Path('../data/processed/split_manifest.json')
if manifest_path.exists():
    print(json.dumps(json.loads(manifest_path.read_text()),indent=2))
else:
    print('Run python scripts/prepare_data.py from the repository root.')


## EDA conclusions (fill in after running)
- Missing values: TODO
- Duplicate rows: TODO
- Class prevalence: TODO
- Temporal variation: TODO
- Boundary ties / duplicate overlap across splits: TODO
- Modeling implications: class imbalance, time order, anonymized features.
